In [ ]:
import os

from aott import PSF_Processing
from aott import Atmosphere_Characterization
from aott import AnalysisViewer
from pathlib import Path
import subprocess
import numpy as np
import pylab as plt

In [ ]:
latest_file = r"C:\Users\foyarzun\Nextcloud\AOTelemetryToolbox\simulation\test.hdf5"

In [ ]:
p2 = PSF_Processing(latest_file, batch_size=200)
# p2.ProcessDark()
p2.SetSkyOrCalibContiditons()
p2.SetPSFModel()
p2.AnalyzeAllTheFile()

In [ ]:
p2.FitPSFModel(p2.long_exp, display=True)

In [ ]:
p2.long_exp_r0_list

In [ ]:
plt.imshow(np.log(np.abs(np.mean(p2.science_frames, axis = 0))))

In [ ]:
plt.imshow(p2.long_exp_psf_model > p2.long_exp_psf_model.max()/2)
plt.colorbar()

In [ ]:
atm_char = Atmosphere_Characterization(latest_file, batch_size=500, filter_TT=False)
atm_char.AnalyzeAllTheFile()

In [ ]:
atm_char.ComputeAtmospericParameters(display=True)

In [ ]:
plt.plot(atm_char.dm_commands[:,0])

In [ ]:
from aott import *
radial_mode_arrays = RadialOrderArray(atm_char.nModes)

GetEstimatedAverageWindspeedZernikeTemporalCutoff(atm_char.dm_Z_modes, atm_char.period, atm_char.loop_leak, radial_mode_arrays, atm_char.Diameter, display = True)

In [ ]:
av = AnalysisViewer(latest_file)

av.CreateAtmosphericAnalysisFigures()
av.CreatePSFAnalysisFigures()

In [ ]:
import subprocess
from datetime import datetime

DATE = "/" + datetime.now().strftime("%Y-%m-%d")
file_title = str(latest_file).split("/")[-1].split(".hdf5")[0]
target_name = file_title.split("-")[0]

cmd = [
    "typst",
    "compile",
    "ao_report.typ",
    "ao_report" + file_title + ".pdf",
    "--input",
    "AOtitle=" + str(latest_file).split("/")[-1].split(".hdf5")[0],
    "--input",
    "telescope=T152-Papyrus",
    "--input",
    "date=" + DATE,
    "--input",
    "target=" + target_name,
    "--input",
    f"elevation={p2.elevation:.1f}",
    "--input",
    "loop_gain=" + str(atm_char.loop_gain),
    "--input",
    "loop_leak=" + str(atm_char.loop_leak),
    "--input",
    "loop_freq=" + str(atm_char.freq),
]

In [ ]:
try:
    result = subprocess.run(
        cmd,
        check=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("STDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)